In [7]:
from pathlib import Path
import sys
import pandas as pd
import io
from typing import Callable
import json
from tqdm.auto import tqdm

# 노트북 기준 상위 폴더를 프로젝트 루트로 가정
PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("현재 경로:", Path.cwd())

from llm.openrouter import chat

현재 경로: c:\Users\user\Desktop\find\compliance_agent\v1_text


In [ ]:
def _strip_json_fence(text: str) -> str:
    cleaned = text.strip()
    if cleaned.startswith("```"):
        lines = cleaned.splitlines()
        if lines and lines[0].startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].startswith("```"):
            lines = lines[:-1]
        cleaned = "\n".join(lines).strip()
    return cleaned

def _extract_json_array_block(text: str) -> str:
    """응답에 설명 문구가 섞여 있어도 JSON 배열 부분만 추출합니다."""
    cleaned = _strip_json_fence(text)
    start = cleaned.find("[")
    if start == -1:
        return cleaned

    depth = 0
    in_string = False
    escape = False

    for i in range(start, len(cleaned)):
        ch = cleaned[i]

        if in_string:
            if escape:
                escape = False
                continue
            if ch == "\\":
                escape = True
            elif ch == '"':
                in_string = False
            continue

        if ch == '"':
            in_string = True
        elif ch == "[":
            depth += 1
        elif ch == "]":
            depth -= 1
            if depth == 0:
                return cleaned[start:i + 1]

    # 끝나는 대괄호를 못 찾으면 가능한 범위만 반환
    return cleaned[start:]

def generate_text(
    dataset_name: str,
    topic: str,
    count: int,
    output_csv_path: str,
    system_prompt: str) -> str:
    user_prompt = f"""
Dataset name: {dataset_name}
Topic: {topic}
Sentence Rows: {count}
""".strip()

    system_prompt = system_prompt.strip()
    data = None
    last_error = None
    last_candidate = ""

    for attempt in range(3):
        retry_hint = ""
        if attempt > 0:
            retry_hint = (
                "\n\n중요: 이전 응답은 JSON 파싱에 실패했습니다. "
                "유효한 JSON 배열만 출력하고, 배열 외 텍스트를 절대 출력하지 마세요."
            )

        raw_response = chat(
            system_prompt=system_prompt,
            user_prompt=user_prompt + retry_hint,
        )
        candidate = _extract_json_array_block(raw_response)
        last_candidate = candidate

        try:
            data = json.loads(candidate)
            break
        except json.JSONDecodeError as exc:
            last_error = exc

    if data is None:
        preview = last_candidate[:300].replace("\n", "\\n")
        raise ValueError(
            f"LLM JSON 파싱 실패: {last_error}. 응답 미리보기: {preview}"
        )

    if not isinstance(data, list):
        raise ValueError("LLM 응답은 JSON 배열(list)이어야 합니다.")

    expected_keys = {"raw_text", "masked_text", "masked_word"}
    normalized_rows = []
    for i, row in enumerate(tqdm(data[:count], desc="rows", unit="row")):
        if not isinstance(row, dict):
            raise ValueError(f"{i}번째 항목이 dict가 아닙니다.")

        missing = expected_keys - set(row.keys())
        if missing:
            raise ValueError(f"{i}번째 항목 키 누락: {missing}")

        normalized_rows.append(
            {
                "raw_text": str(row["raw_text"]),
                "masked_text": str(row["masked_text"]),
                "masked_word": str(row["masked_word"]),
            }
        )

    df = pd.DataFrame(normalized_rows, columns=["raw_text", "masked_text", "masked_word"])

    output_path = Path(output_csv_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_path, index=False, encoding="utf-8-sig")

    return str(output_path)

In [ ]:
system_prompt = """

** Instructions **
- 당신은 한국어 문장에서 개인정보, 금융정보, 거래정보, 인증정보를 식별하고 마스킹하는 데이터 생성기다.
- 입력 문장을 그대로 보존하되, 마스킹 대상에 해당하는 문자열만 지정된 변수명 태그로 치환한다.
- 출력은 반드시 'raw_text'와 'masked_text', 'masked_word' 세 필드를 가지는 단일 JSON 객체 1개만 생성한다.
- 'raw_text'에는 원문을 그대로 넣는다. 띄어쓰기, 문장부호, 조사, 어미를 임의로 수정하지 않는다.
- 'masked_text'에는 원문 구조와 순서를 유지한 채 민감정보만 치환한 결과를 넣는다.
- 'masked_word'에는 마스킹된 단어를 넣는다.
- 태그 형식은 반드시 `[VARIABLE_NAME]` 형식을 사용한다.
- 같은 유형의 민감정보가 여러 번 나오면 모두 같은 태그로 각각 치환한다.
- 하나의 표현이 여러 의미로 해석될 수 있으면 문맥상 가장 직접적인 민감정보 유형 하나만 선택한다.
- 설명, 해설, 주석, 코드블록, 접두어, 접미어 없이 JSON만 출력한다.
- 존재하지 않는 변수명은 만들지 않는다.
- 값 일부만 남기거나 부분 마스킹하지 않는다. 식별된 전체 표현을 통째로 치환한다.
- 사람 이름은 호칭, 조사, 직함을 제외하고 이름 부분만 치환한다. 예: '김철수 고객님' -> '[PERSON_NAME] 고객님'
- 주소, 상호명, 거래처명, 고객ID, 인증서 정보 등은 문맥상 해당 항목이 명확할 때만 치환한다.
- 금액, 잔액, 거래 관련 수치도 문맥상 금융 거래 정보일 때만 치환한다.

** VARIABLE_NAME **
- PERSON_NAME: 사람 이름
- RESIDENT_ID: 주민등록번호
- BIRTH_DATE: 생년월일
- PHONE_NUMBER: 전화번호
- DRIVER_LICENSE: 운전면허번호
- ADDRESS: 주소
- EMAIL: 이메일
- ACCOUNT_NUMBER: 계좌번호
- CARD_NUMBER: 카드번호
- CUSTOMER_ID: 고객ID
- CARD_CVC: 카드 CVC
- ACCOUNT_PASSWORD: 계좌 비밀번호
- CERTIFICATE_ID: 인증서 정보
- TRANSACTION_AMOUNT: 거래금액
- MERCHANT_NAME: 거래 내역 또는 가맹점명
- ACCOUNT_BALANCE: 잔액
- OTP_CODE: OTP 코드
- AUTH_CODE: 인증번호
- PASSWORD: 비밀번호
- ACCESS_TOKEN: Access Token

output format
{
  "raw_text": "<입력 원문 그대로>",
  "masked_text": "<민감정보만 [VARIABLE_NAME]으로 치환한 문장>",
  "masked_word": "<마스킹된 단어>"
}

"""

output_file = generate_text(
    dataset_name="금융 고객 상담 마스킹",
    topic="금융 고객 상담에서 개인정보를 마스킹한 텍스트 데이터셋",
    count=2,
    output_csv_path=f"data/masked_{dataset_name}_{}.csv",
    system_prompt=system_prompt
)

print(f"생성된 파일: {output_file}")
preview_df = pd.read_csv(output_file)
preview_df.head()

생성된 파일: data\masking_customer_support.csv


,raw_text,masked_text,masked_word
0,"안녕하세요, 고객님. 귀하의 계좌 번호는 1234-5678-9012입니다.","안녕하세요, 고객님. 귀하의 계좌 번호는 ****-****-****입니다.",1234-5678-9012
1,"고객님의 이름은 홍길동이며, 전화번호는 010-1234-5678입니다.","고객님의 이름은 ****이며, 전화번호는 010-****-5678입니다.","홍길동, 1234"


In [ ]:
!pip install datetime

In [ ]:
import datetime
date_str = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
date_str